# Part 13: 캡스톤 프로젝트**소요 시간**: 약 2.5시간**난이도**: ⭐⭐⭐⭐⭐ (고급)**형태**: 워크숍 (튜토리얼이 아닌 실습 가이드)**목표**: Part 1-12에서 배운 모든 기술을 통합하여 나만의 GraphRAG 시스템을 엔드투엔드로 구축한다.---> **이 노트북은 대부분 TODO입니다.** 여러분이 직접 코드를 작성해야 합니다.> Part 1-12의 코드를 참고하면서 진행하세요.---## 워크숍 구조| 섹션 | 시간 | 내용 ||------|------|------|| 1. 킥오프 | 20분 | 도메인 선택, 아키텍처 설계, 예산 추정 || 2. 구축 | 70분 | 데이터 수집 → 온톨로지 → 추출 → 적재 → 검색 || 3. 평가 | 30분 | RAGAS 평가, 비용/속도/정확도 리포트 || 4. 발표 | 30분 | 라이브 데모, 아키텍처 설명, 회고 |

---## 1. 환경 설정### 패키지 설치 및 Neo4j 연결

In [ ]:
import os, json, timefrom dotenv import load_dotenvfrom neo4j import GraphDatabaseload_dotenv()load_dotenv(dotenv_path="../.env")NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))driver.verify_connectivity()print("[OK] 환경 준비 완료")def run_query(query, parameters=None, print_result=True):    with driver.session() as session:        result = session.run(query, parameters or {})        records = [record.data() for record in result]        if print_result:            for i, rec in enumerate(records, 1):                print(f"  [{i}] {rec}")        return records

------# Section 1: 프로젝트 킥오프 (20분)## 1.1 도메인 선택아래 도메인 중 하나를 선택하거나, 본인 도메인을 직접 정의하세요.| 도메인 | 데이터 소스 | 난이도 | 핵심 관계 ||--------|-----------|--------|----------|| 금융/투자 | 뉴스, 공시 | ⭐⭐⭐ | 투자, 경쟁, 인수 || 법률 | 판례, 법령 | ⭐⭐⭐⭐ | 인용, 적용, 해석 || 의료 | 논문, 가이드라인 | ⭐⭐⭐⭐⭐ | 치료, 부작용, 상호작용 || IT/통신 | 기술 문서 | ⭐⭐⭐ | 의존, 호환, 대체 || 자유 선택 | - | - | - |

In [ ]:
# TODO: 도메인 정의MY_DOMAIN = ""  # 예: "금융/투자"DATA_SOURCE = ""  # 예: "한국 IT 기업 뉴스 기사 20개"KEY_QUESTION = ""  # 예: "삼성전자에 투자한 기관이 투자한 다른 기업은?"print("=== 프로젝트 도메인 ===")print(f"  도메인: {MY_DOMAIN or '미정'}")print(f"  데이터: {DATA_SOURCE or '미정'}")print(f"  핵심 질문: {KEY_QUESTION or '미정'}")if all([MY_DOMAIN, DATA_SOURCE, KEY_QUESTION]):    print("\n[OK] 도메인 정의 완료!")else:    print("\n[TODO] 위 변수를 채우세요!")

## 1.2 아키텍처 설계### TODO: 5-Layer 아키텍처를 설계하세요```L1 Strategy   : 도메인 특성 → GraphRAG 도입 판단L2 Data       : 데이터 수집 → 전처리 → BaselineL3 Infra      : Neo4j 설정, 인덱스, 메모리L4 Processing : 온톨로지 → LLM 추출 → ER → 적재 → 검색L5 Deployment : 평가, 최적화, 모니터링```

In [ ]:
# TODO: 아키텍처 정의architecture = {    "L1_strategy": {        "domain": MY_DOMAIN,        "why_graphrag": "",  # 예: "Multi-hop 투자 관계 추적 필요"    },    "L2_data": {        "source": DATA_SOURCE,        "doc_count": 0,        "preprocessing": [],  # 예: ["PDF→텍스트", "청크 분할"]    },    "L3_infra": {        "neo4j_version": "5-community",        "plugins": ["apoc"],    },    "L4_processing": {        "ontology_entities": [],   # 예: ["Company", "Person", "Product"]        "ontology_relations": [],  # 예: ["INVESTS_IN", "DEVELOPS"]        "llm_model": "gpt-4o",        "search_type": "hybrid",   # text2cypher, vector, hybrid    },    "L5_deployment": {        "eval_method": "RAGAS",        "target_accuracy": 0.7,    }}# TODO: 위 딕셔너리를 채우세요print("아키텍처를 설계해주세요!")

In [ ]:
# TODO: 예산 추정def estimate_budget(num_docs, num_queries_per_day, model="gpt-4o"):    prices = {        "gpt-4o": {"input": 0.005, "output": 0.015},        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},    }    p = prices.get(model, prices["gpt-4o"])    # 인덱싱 비용 (1회)    tokens_per_doc = 2000    indexing_cost = num_docs * tokens_per_doc / 1000 * (p["input"] + p["output"])    # 월간 쿼리 비용    tokens_per_query = 3000    monthly_queries = num_queries_per_day * 30    query_cost = monthly_queries * tokens_per_query / 1000 * (p["input"] + p["output"])    print(f"=== 예산 추정 ({model}) ===")    print(f"  문서 수: {num_docs}개")    print(f"  일일 쿼리: {num_queries_per_day}회")    print(f"  인덱싱 비용 (1회): ${indexing_cost:.2f}")    print(f"  월간 쿼리 비용: ${query_cost:.2f}")    print(f"  월간 총비용: ${indexing_cost + query_cost:.2f}")# TODO: 본인 프로젝트에 맞게 수정# estimate_budget(num_docs=50, num_queries_per_day=100, model="gpt-4o-mini")print("예산 추정 함수를 실행해보세요!")

------# Section 2: 엔드투엔드 구축 (70분)## TODO: 각 단계를 구현하세요| 단계 | 참고 Part | 예상 시간 ||------|----------|----------|| 데이터 수집 | Part 2 | 10분 || 온톨로지 설계 | Part 2 | 10분 || LLM 추출 | Part 3 | 15분 || Entity Resolution | Part 4 | 10분 || Neo4j 적재 | Part 1-2 | 10분 || 검색 파이프라인 | Part 6, 10 | 15분 |

In [ ]:
# TODO: 1. 데이터 수집 (Part 2 참고)# documents = load_documents("./data/my_docs/")print("[TODO] 데이터를 수집하세요")

In [ ]:
# TODO: 2. 온톨로지 설계 (Part 2 참고)ontology = {    "entity_types": [        # {"name": "Company", "description": "기업"},    ],    "relation_types": [        # {"name": "INVESTS_IN", "source": "Investor", "target": "Company"},    ]}print("[TODO] 온톨로지를 정의하세요")

In [ ]:
# TODO: 3. LLM 추출 (Part 3 참고)# from openai import OpenAI# client = OpenAI()# extracted = extract_entities_relations(text, ontology)print("[TODO] LLM으로 엔티티/관계를 추출하세요")

In [ ]:
# TODO: 4. Entity Resolution (Part 4 참고)# from rapidfuzz import fuzz# duplicates = find_duplicates(entities, threshold=0.85)print("[TODO] 중복 엔티티를 해소하세요")

In [ ]:
# TODO: 5. Neo4j 적재 (Part 1-2 참고)# for entity in entities:#     run_query("MERGE (n:Entity {name: $name}) SET n.type = $type",#               {"name": entity["name"], "type": entity["type"]})print("[TODO] Neo4j에 적재하세요")

In [ ]:
# TODO: 6. 검색 파이프라인 (Part 6, 10 참고)# def graphrag_search(question: str):#     cypher = generate_cypher(question, schema)#     results = run_query(cypher)#     answer = generate_answer(question, results)#     return answerprint("[TODO] 검색 파이프라인을 구현하세요")

------# Section 3: 평가 + 벤치마크 (30분)

In [ ]:
# TODO: RAGAS 평가 데이터셋 (30개 질문)eval_data = [    # Easy (1-hop) - 10개    # {"question": "???", "ground_truth": "???", "difficulty": "easy"},    # Medium (2-hop) - 10개    # {"question": "???", "ground_truth": "???", "difficulty": "medium"},    # Hard (Multi-hop) - 10개    # {"question": "???", "ground_truth": "???", "difficulty": "hard"},]print(f"평가 데이터: {len(eval_data)}개")print("[TODO] 30개 질문을 작성하세요!")

In [ ]:
# TODO: RAGAS 평가 실행# from ragas import evaluate# from ragas.metrics import faithfulness, answer_relevancy# results = evaluate(dataset, metrics=[faithfulness, answer_relevancy])print("[TODO] RAGAS 평가를 실행하세요 (Part 7 참고)")

In [ ]:
# TODO: 비용/속도/정확도 리포트# print("=== 벤치마크 리포트 ===")# print(f"정확도: {accuracy:.1%}")# print(f"평균 응답시간: {avg_latency:.2f}초")# print(f"질문당 비용: ${cost_per_query:.4f}")print("[TODO] 벤치마크 리포트를 생성하세요")

------# Section 4: 발표 + 피드백 (30분)## 데모 체크리스트| 항목 | 체크 ||------|------|| Neo4j 실행 확인 | [ ] || 데이터 로드 확인 | [ ] || API 키 확인 | [ ] || Easy 질문 테스트 | [ ] || Medium 질문 테스트 | [ ] || Hard 질문 테스트 | [ ] |## 발표 구조 (15분)1. **문제 정의** (2분): 도메인, GraphRAG 필요성2. **아키텍처** (3분): 5-Layer 다이어그램3. **라이브 데모** (3분): Easy/Medium/Hard 질문4. **벤치마크** (2분): 정확도/속도/비용5. **Q&A** (5분)

In [ ]:
# TODO: 데모 질문 선정demo_queries = {    "easy": "",     # 1-hop 질문    "medium": "",   # 2-hop 질문    "hard": "",     # Multi-hop 질문}for difficulty, query in demo_queries.items():    print(f"{difficulty.upper():10} : {query or '미정'}")if all(demo_queries.values()):    print("\n[OK] 데모 준비 완료!")else:    print("\n[TODO] 데모 질문을 선정하세요!")

---## 회고### 캡스톤 프로젝트를 마치며1. **가장 어려웠던 부분은?**   -2. **가장 인상 깊었던 기법은?**   -3. **실무에 적용할 아이디어는?**   -4. **추가 학습하고 싶은 주제는?**   ----### Part 1-13 전체 여정| Part | 주제 | Milestone ||------|------|----------|| 1-7 | 기초 GraphRAG | 온톨로지 → LLM 추출 → 검색 → 평가 || 8-11 | 고급 주제 | 프레임워크, 알고리즘, Agent, 디버깅 || 12 | 엔터프라이즈 | PoC, 보안, CI/CD || **13** | **캡스톤** | **엔드투엔드 시스템 완성** |> **이제 여러분은 GraphRAG를 리드할 수 있습니다.**> 깊이가 곧 가치입니다. 실무에 적용하고, 커뮤니티에 기여하세요!

In [ ]:
# 세션 정리# driver.close()print("=" * 60)print("  Part 13 캡스톤 프로젝트를 마칩니다.")print("  수고하셨습니다! GraphRAG 여정을 계속 이어가세요!")print("=" * 60)